In [1]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
print("Connected. Ready to query.")

Paste your Hugging Face READ token (hf_...): ··········
Connected. Ready to query.


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anish494/flyrank_ai_first_assignment/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item, as of a single mid-panel month (month=2026-03).**

For clustering, I need one row per page describing its typical behavior during that
month — not a full daily time series. I'll aggregate `fact_content_daily_performance`
down to one row per `content_hash_id` for March 2026 only, verified below.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Features** (the five behavioral signals I'll cluster on, aggregated for March 2026):
- `gsc_impressions` (summed for the month) — real search visibility
- `gsc_clicks` (summed for the month) — real engagement
- `gsc_avg_position` (averaged for the month) — real ranking position
- `word_count` (from `dim_content`, static metadata) — content depth
- `content_age_days` (from `dim_content`, computed at query time) — content lifecycle stage

**Label/proxy:** none in the traditional sense — clustering has no ground-truth label.
The closest equivalent is the cluster assignment itself, produced by the algorithm, not
observed in the data.

**Context (used for grouping/joining, not modeling):**
- `client_hash_id`, `content_hash_id` — join keys, carry no behavioral meaning themselves
- `report_date` — used only to filter to the March 2026 window, not as a feature

**Excluded (deliberately):**
- `trend_pct` and `trend_direction` — these describe *change over time*, which is
  exactly the kind of definitionally-loaded quantity I want to test as a deliberate
  leak in Section 3, rather than use as an honest feature.
- Any GA4 engagement fields (`sessions`, `scroll_rate`) — excluded to keep this contract
  focused on search-only signals for now, and because GA4 coverage is inconsistent
  across clients (some have `ga4_data_start` far later than `gsc_data_start`, or missing
  entirely), which would bias a clustering feature toward clients with better tracking.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries below, run against `month=2026-03` (a mid-panel month, never the
`_sample` table): grain, row count + date span, and availability with `IS TRUE`.
Then a five-feature frame, then the deliberate leak experiment.

In [2]:
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Grain check: for March 2026, is content_hash_id unique after aggregation?
grain_check = con.sql(f"""
    WITH march AS (
        SELECT content_hash_id
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
        GROUP BY content_hash_id
    )
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT content_hash_id) AS distinct_content_items
    FROM march
""").df()

print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  distinct_content_items
0      331437                  331437


In [3]:
span_check = con.sql(f"""
    SELECT COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(*) AS n_daily_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

print(span_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_content_items   min_date   max_date  n_daily_rows
0           331437 2026-03-01 2026-03-31       9841378


In [4]:
# Availability check: how many March rows actually have GA4 tracking active?
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ga4  pct_with_ga4
0     9841378       413966.0           4.2


**Confirms Section 2's exclusion decision with a real number:** only 4.2% of March
2026 rows have `ga4_data_available IS TRUE`. Including GA4 engagement fields as
clustering features would mean the vast majority of pages have missing/zero values
for those features — not a genuine behavioral signal, just an artifact of tracking
coverage. This is why GA4 fields stay excluded from this contract.

In [6]:
# Check what columns dim_content actually has
schema_check = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
print(schema_check.to_string())

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [7]:
# Five-feature frame: one row per content item, aggregated over March 2026
features_df = con.sql(f"""
    SELECT
        f.content_hash_id,
        SUM(f.gsc_impressions)      AS impressions_mar,
        SUM(f.gsc_clicks)           AS clicks_mar,
        AVG(f.gsc_avg_position)     AS avg_position_mar,
        ANY_VALUE(c.word_count)     AS word_count,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-31') AS content_age_days
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
    GROUP BY f.content_hash_id
""").df()

print(f"{len(features_df):,} content items with a March feature row")
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331,437 content items with a March feature row


,content_hash_id,impressions_mar,clicks_mar,avg_position_mar,word_count,content_age_days
0,content_0236ef736698e17c,0.0,0.0,NaN,4335,175
1,content_025f6cfd3c298870,0.0,0.0,NaN,3719,175
2,content_0263d5f9b7a2ecd4,1.0,0.0,9.0,3246,175
3,content_02752c6c1c60161f,0.0,0.0,NaN,3641,175
4,content_0317b24cc1ff5c5d,0.0,0.0,NaN,4118,184


**Five features, each with an "available when?" line:**

1. **`impressions_mar`** (summed GSC impressions for March) — knowable at the decision
   moment because it's a direct historical measurement of what already happened; no
   future information involved.
2. **`clicks_mar`** (summed GSC clicks for March) — same reasoning: a historical count
   of actual events that already occurred by the end of the analysis window.
3. **`avg_position_mar`** (averaged GSC position for March) — knowable because it's the
   observed average of a page's actual search ranking during the window, not a
   prediction of future ranking.
4. **`word_count`** — knowable because it's a static property of the published content
   itself, fixed independent of any search performance outcome.
5. **`content_age_days`** — knowable because it's computed purely from
   `content_created_date`, a fact that existed before the March window even began.

**Data quality note:** rows with 0 impressions correctly show `NaN` for
`avg_position_mar`, since there's no position to average when a page received no
impressions. This needs handling before clustering (e.g. filtering to
`impressions_mar > 0`, matching the same discipline used in the starter pipeline).

In [9]:
# --- THE TRAP: deliberately add a label-derived / future-leaking column ---
import pandas as pd
# Pull April impressions too (the "future" relative to our March window)
april_check = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_apr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-04-01' AND report_date < '2026-05-01'
    GROUP BY content_hash_id
""").df()

leaky_df = features_df.merge(april_check, on="content_hash_id", how="left")
leaky_df["impressions_apr"] = leaky_df["impressions_apr"].fillna(0)

# The leaky "feature": % change from March to April -- this is FUTURE information
# relative to our March decision window, and it directly encodes what a "declining"
# vs "growing" cluster would look like before we've done any real clustering.
leaky_df["pct_change_mar_to_apr"] = (
    (leaky_df["impressions_apr"] - leaky_df["impressions_mar"]) /
    leaky_df["impressions_mar"].replace(0, pd.NA)
)

# Show how suspiciously clean a "declining vs growing" split becomes using this leak
declining = (leaky_df["pct_change_mar_to_apr"] < -0.2).sum()
growing   = (leaky_df["pct_change_mar_to_apr"] > 0.2).sum()
print(f"Using the leaked feature, {declining:,} pages look 'declining' and {growing:,} look 'growing'")
print("\nBut this 'feature' is just April's outcome relative to March -- it doesn't")
print("describe March behavior at all. It would let any grouping method trivially")
print("separate pages by an outcome that hadn't happened yet at decision time.")

# Now: DELETE the leak and keep the honest frame
leaky_df = leaky_df.drop(columns=["impressions_apr", "pct_change_mar_to_apr"])
print("\nLeak removed. Honest feature frame restored:")
print(leaky_df.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Using the leaked feature, 94,000 pages look 'declining' and 48,920 look 'growing'

But this 'feature' is just April's outcome relative to March -- it doesn't
describe March behavior at all. It would let any grouping method trivially
separate pages by an outcome that hadn't happened yet at decision time.

Leak removed. Honest feature frame restored:
['content_hash_id', 'impressions_mar', 'clicks_mar', 'avg_position_mar', 'word_count', 'content_age_days']


**The trap, summarized:** `pct_change_mar_to_apr` produced a clean 94,000 vs 48,920
split — but that split is entirely manufactured from information (April's outcome)
that didn't exist at the March decision point. This is the same leakage lesson from
Notebook 02's `trend_pct` demo, reproduced here on the real warehouse: a feature that
encodes the future (or the label) will always look like a great signal and mean
nothing. The honest five-feature frame above contains only March-observable signals.

## 4. Data limits

**One named limitation of this slice:** GA4 engagement coverage is severely limited in
March 2026 — only 4.2% of daily rows have `ga4_data_available IS TRUE` (verified in
Section 3). This means any archetype I build from this month's data reflects search
behavior almost exclusively, not true user engagement (scroll depth, session quality,
etc.) for the vast majority of pages. An archetype like "high traffic, low CTR" is
trustworthy; an archetype claiming anything about on-page engagement would not be,
since the underlying data simply isn't there for 95.8% of content items this month.

Additionally, this is a single mid-panel month (March 2026) chosen deliberately to
avoid the `_sample` table's future-leakage risk — but that also means this slice
cannot speak to seasonality or how stable these archetypes are across different
months. A page's cluster membership in March isn't guaranteed to hold in, say,
December, and this contract makes no claim that it does.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.